# Wave propagation through a soil block with an embedded crack

A 100 m cubic block of linear-elastic soil with a 20 m square crack
embedded in the XY plane at *z* = 0.  The crack is fragmented into the
solid (`cleanup_free=False` keeps embedded interior surfaces alive),
then split into two coincident face entities by the Gmsh `Crack` plugin
via `g.mesh.editing.crack(... side_labels=True)`, which auto-creates
the physical groups `Crack_normal` and `Crack_inverted`.

**Loading.** Symmetric internal pressure on the two crack faces
(`surface(magnitude=+P, normal=True)` on both `Crack_normal` and
`Crack_inverted`).  Because the resolver uses the **per-face physical
outward normal** (computed from adjacent-tet centroids), the same
positive magnitude on both coincident faces produces opposite-direction
nodal forces — pushing each face into its own bonded body interior
and prying the crack open.  The pattern uses a **`Path` time series
ramp** from 0 to 1 over `T_ramp`, held at 1 to `T_total`.

**Two-stage analysis.** Stage 1 captures the loading transient as the
ramp builds; stage 2 holds the load and watches reflections off the
fully-fixed outer boundaries.  Newmark `(γ=0.5, β=0.25)`.

**Recording.** Vanilla `ops.recorder("mpco", ...)` capturing nodal
displacement / velocity / acceleration plus element stresses /
strains every step.  Read back via `Results.from_mpco` and shown in
the post-solve viewer.

**Strategy.** apeGmsh on the geometry/mesh side; **vanilla openseespy
for the live model** (nodes/elements/materials/fixity emitted from
the FEMData broker via plain `ops.*` calls).  apeGmsh's
`g.opensees.*` composite is used in parallel only to write a
model-only TCL for archival.

**Outputs.** All files land in `wave_propagation_out/` next to the
notebook:
* `wave_propagation.mpco` — recorder output
* `wave_propagation_model.tcl` — model-only TCL export

> ⚠️ Boundary reflections.  All six outer faces are fully fixed per
> spec, so waves bounce back rather than radiate.  For a real wave
> study you'd want absorbing boundaries (PML / Lysmer dashpots).


## 1. Imports and parameters

In [ ]:
from pathlib import Path

import numpy as np
import openseespy.opensees as ops

from apeGmsh import Results, apeGmsh

# Output directory next to the notebook
HERE = Path.cwd()
OUT  = HERE / "wave_propagation_out"
OUT.mkdir(exist_ok=True)
MPCO_FILE = OUT / "wave_propagation.mpco"
TCL_FILE  = OUT / "wave_propagation_model.tcl"

# Geometry  (m)
BOX_SIZE   = 100.0
PLANE_SIZE = 20.0
MESH_SIZE  = 5.0

# Material  (SI)
E_SOIL  = 100.0e6     # Pa
NU_SOIL = 0.30
RHO     = 2000.0      # kg/m^3

# Loading
P_PEAK  = 1.0e5       # Pa, internal pressure applied to both crack faces

# Time
DT      = 1.0e-3
T_RAMP  = 0.10
T_TOTAL = 0.50


## 2. Geometry — box with embedded crack plane

Add the box and the rectangle, fragment them so the rectangle is
embedded as a conformal interior surface.  `cleanup_free=False`
prevents the post-fragment cleanup from sweeping the embedded face
(which has no upward volume adjacency and would otherwise be treated
as a stray cutting-plane remnant).


In [ ]:
m = apeGmsh(model_name="wave_crack", verbose=True)
m.begin()

m.model.geometry.add_box(
    -BOX_SIZE / 2, -BOX_SIZE / 2, -BOX_SIZE / 2,
    BOX_SIZE, BOX_SIZE, BOX_SIZE, label="box",
)
m.model.geometry.add_rectangle(
    -PLANE_SIZE / 2, -PLANE_SIZE / 2, 0.0,
    PLANE_SIZE, PLANE_SIZE, label="plane",
)

m.model.boolean.fragment(objects="box", tools="plane", cleanup_free=False)


## 3. Physical groups — body, crack, outer boundary

`Outer` is defined as the boundary of the box volume.  The embedded
crack face is *not* in the volume's BRep boundary, so this query
cleanly returns just the six outer cube faces.


In [ ]:
m.physical.add_volume("box", name="Body")
m.physical.add_surface("plane", name="Crack")
m.physical.add_surface(
    m.model.queries.boundary("box", dim=2), name="Outer",
)


## 4. Mesh, then duplicate plane-interior nodes

Mesh first, then call the Crack plugin.  `side_labels=True` (default)
auto-creates physical groups `Crack_normal` and `Crack_inverted` on
the two surviving face entities, classified by which side of the
original surface normal their adjacent volume tets sit on.  RCM
renumbering gives OpenSees dense 1-based IDs and reduces bandwidth.


In [ ]:
m.mesh.sizing.set_global_size(MESH_SIZE)
m.mesh.generation.generate(dim=3)

m.mesh.editing.crack("Crack", dim=2)

m.mesh.partitioning.renumber(dim=3, method="rcm", base=1)


## 5. Loads — symmetric internal pressure on the two crack faces

Apply the same positive pressure `+P_PEAK` to **both** `Crack_normal`
and `Crack_inverted`.  The composite resolves a per-face physical
outward normal from the adjacent-tet centroid (instead of the
connectivity-derived normal, which would be the same `+z` on both
coincident entities).  Because each face's outward points *into* its
own bonded body's exterior, the convention `f3 = -P · A · outward`
maps a positive pressure to nodal forces that push **into** each
body's interior:

| Face            | Physical outward | Nodal force direction |
| :-------------- | :--------------: | :-------------------: |
| `Crack_normal`   |       `-z`       |        `+z`           |
| `Crack_inverted` |       `+z`       |        `-z`           |

Net effect: the two coincident faces are driven apart, prying the
crack open.  Resolution into `NodalLoadRecord`s happens when we
extract the FEMData snapshot below.

> Sign mnemonic:  `surface(magnitude=+P, normal=True)` is a pressure
> load (positive = compression into the bonded body).  For a crack,
> "into each body" means "away from the other crack face" — so the
> same positive sign on both faces opens the crack.  Use a negative
> magnitude on both for a suction load that closes the crack.


In [ ]:
with m.loads.pattern("CrackPressure"):
    m.loads.surface("Crack_normal",   magnitude=+P_PEAK, normal=True)
    m.loads.surface("Crack_inverted", magnitude=+P_PEAK, normal=True)


## 6. Optional — model-only TCL export via the apeGmsh OpenSees composite

apeGmsh's `g.opensees.*` declares the model in its internal tables
and writes a TCL via `export.tcl(...)`.  This path **does not**
populate the live openseespy domain — it's an archival-format export.
The live model is built from FEMData with vanilla `ops.*` calls
below.


In [ ]:
m.opensees.set_model(ndm=3, ndf=3)
m.opensees.materials.add_nd_material(
    "Soil", "ElasticIsotropic", E=E_SOIL, nu=NU_SOIL, rho=RHO,
)
m.opensees.elements.assign("Body", "FourNodeTetrahedron", material="Soil")
m.opensees.elements.fix("Outer", dofs=[1, 1, 1])

m.opensees.build()
m.opensees.export.tcl(str(TCL_FILE))
print(f"wrote {TCL_FILE}")


## 7. Build the live OpenSees model — vanilla openseespy

Pull nodes / elements / surface-load records from the FEMData broker
and emit `ops.*` directly.  This is the canonical apeGmsh-→-OpenSees
pattern (matches the `plate_with_crack` example).


In [ ]:
fem = m.mesh.queries.get_fem_data(dim=3)
print(fem.inspect.summary())

ops.wipe()
ops.model("basic", "-ndm", 3, "-ndf", 3)

# Nodes
for nid, xyz in fem.nodes.get():
    ops.node(int(nid), float(xyz[0]), float(xyz[1]), float(xyz[2]))

# Material
MAT_TAG = 1
ops.nDMaterial("ElasticIsotropic", MAT_TAG, E_SOIL, NU_SOIL, RHO)

# Elements — one FourNodeTetrahedron per Body tet
n_elems = 0
for group in fem.elements.get(pg="Body"):
    for eid, conn in zip(group.ids, group.connectivity):
        ops.element(
            "FourNodeTetrahedron", int(eid),
            int(conn[0]), int(conn[1]), int(conn[2]), int(conn[3]),
            MAT_TAG,
        )
        n_elems += 1

# Fix every node on the outer boundary in all three translations
outer_ids = fem.nodes.get(pg="Outer").ids
for n in outer_ids:
    ops.fix(int(n), 1, 1, 1)

print(f"live model: {len(ops.getNodeTags())} nodes, {n_elems} tets, "
      f"{len(outer_ids)} outer-fixed nodes")


## 8. Time series + pattern + nodal loads (raw openseespy)

`Path` series: factor = 0 at *t* = 0, ramps to 1 by `T_ramp`, holds
at 1 through `T_total`.  Per-node `ops.load` calls come from the
resolved `surface()` records in the FEM broker.


In [ ]:
ops.timeSeries(
    "Path", 1,
    "-time",   0.0,    T_RAMP, T_TOTAL,
    "-values", 0.0,    1.0,    1.0,
)
ops.pattern("Plain", 1, 1)

n_loaded = 0
for nl in fem.nodes.loads:
    if nl.force_xyz is None:
        continue
    fx, fy, fz = nl.force_xyz
    ops.load(int(nl.node_id), float(fx), float(fy), float(fz))
    n_loaded += 1
print(f"applied {n_loaded} nodal loads")


## 9. MPCO recorder — vanilla `ops.recorder`

Single MPCO file capturing nodal kinematic fields and per-Gauss-point
stress / strain on every tet.  `-T dt DT` records every analysis step
(every `DT` of model time).


In [ ]:
if MPCO_FILE.exists():
    MPCO_FILE.unlink()

ops.recorder(
    "mpco", str(MPCO_FILE),
    "-N", "displacement", "velocity", "acceleration",
    "-E", "stresses", "strains",
    "-T", "dt", DT,
)


## 10. Two-stage transient analysis

Standard `Newmark(γ=0.5, β=0.25)` with `BandGeneral` solver.  Stage 1
runs `T_ramp / DT` steps to capture the loading transient; stage 2
runs the remaining steps with the load held to capture reflections
off the fixed boundaries.  Returns from `analyze` are checked
explicitly so a divergence aborts cleanly.


In [ ]:
ops.constraints("Plain")
ops.numberer("RCM")
ops.system("BandGeneral")
ops.test("NormDispIncr", 1.0e-6, 50)
ops.algorithm("Newton")
ops.integrator("Newmark", 0.5, 0.25)
ops.analysis("Transient")

# Stage 1 — load ramp (0 to T_RAMP)
n_steps_ramp = int(round(T_RAMP / DT))
print(f"[stage 1] ramping load over {T_RAMP:.3f} s "
      f"({n_steps_ramp} steps of dt={DT})")
ok = ops.analyze(n_steps_ramp, DT)
assert ok == 0, f"stage 1 failed (analyze returned {ok})"

# Stage 2 — load held, capture reflections
n_steps_hold = int(round((T_TOTAL - T_RAMP) / DT))
print(f"[stage 2] holding load for {T_TOTAL - T_RAMP:.3f} s "
      f"({n_steps_hold} steps of dt={DT})")
ok = ops.analyze(n_steps_hold, DT)
assert ok == 0, f"stage 2 failed (analyze returned {ok})"


## 11. Read results back from the .mpco file

Wipe the analysis and remove the recorders so the file handle is
flushed before `Results.from_mpco` opens it.


In [ ]:
ops.wipeAnalysis()
ops.remove("recorders")

results = Results.from_mpco(MPCO_FILE, fem=fem)
print(results.inspect.summary())


## 12. Open the post-solve viewer

Geometry → Diagram → Layer hierarchy.  Use the scrubber to walk the
500 frames; deformed shape is per-Geometry; switch contour fields
between displacement / velocity / acceleration / stress / strain via
the side rail.


In [ ]:
results.viewer()


In [ ]:
m.end()
